Google Colab에서 데이터 전처리부터 모델 학습, 평가, 그리고 로컬 데스크탑 서빙을 위한 ONNX 변환 및 다운로드까지의 **전체 엔드투엔드 개발 과정**입니다.

이 과정을 Colab 노트북(`.ipynb`)의 셀 단위로 구성하여 그대로 복사해 순서대로 실행할 수 있도록 정리했습니다.

---

### [Step 1] Google Drive 연동 및 필수 라이브러리 설치

Colab 상단 메뉴에서 **[런타임] $\rightarrow$ [런타임 유형 변경] $\rightarrow$ [T4 GPU] 또는 고성능 CPU**를 선택한 후 시작합니다.

In [1]:
# [Cell 1] 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# [Cell 2] 필요한 의존성 패키지 설치
!pip install -q lightgbm optuna onnx onnxruntime onnxmltools scikit-learn

# Python 3.13 / numpy._core 호환성 우회를 위한 동적 핫패치(Mocking)
import sys
import numpy as np

try:
    if not hasattr(np, '_core'):
        import numpy.core as _core
        np._core = _core
    if not hasattr(np._core, '_multiarray_umath'):
        import numpy._core._multiarray_umath as _multiarray_umath
    if not hasattr(np._core._multiarray_umath, '_blas_supports_fpe'):
        np._core._multiarray_umath._blas_supports_fpe = lambda x: False
except Exception:
    pass

import optuna
import onnxmltools
print("필수 라이브러리 설치 및 정상 로드 확인 완료!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
필수 라이브러리 설치 및 정상 로드 확인 완료!


---

### [Step 2] 원천 데이터 로드 및 결측치 도메인 정제

구글 드라이브 내에 `2025_서울25개구_시간대별_교통_날씨_통합_01-23시.csv` 파일을 미리 업로드해 둔 경로를 지정합니다.

In [2]:
# [Cell 3] 데이터 로드 및 초기 정제
import pandas as pd
import numpy as np

# 드라이브 내 실제 파일 경로 입력
CSV_PATH = '/content/drive/MyDrive/weather-transit-web/data/2025_서울25개구_시간대별_교통_날씨_통합_01-23시.csv'
df = pd.read_csv(CSV_PATH)
print(f"원천 데이터 로드 완료: {df.shape}")

# 1. 지하철 심야 운휴 시간대(01~05시) 명시적 0 처리 (사양서 준수)
df.loc[df['시간'].isin([1, 2, 3, 4, 5]), ['지하철_승차', '지하철_하차']] = 0

# 2. 기상 결측치 처리 (강수는 0, 기온/풍속/습도는 선형 보간)
df['강수량(mm)'] = df['강수량(mm)'].fillna(0.0)
df['기온(°C)'] = df['기온(°C)'].interpolate(method='linear')
df['풍속(m/s)'] = df['풍속(m/s)'].interpolate(method='linear')
df['습도(%)'] = df['습도(%)'].interpolate(method='linear')

# 3. 버스 결측 일자 보정 (전후 동일 요일/시간대 기준 대체 또는 선형 보간)
df['버스_승차'] = df['버스_승차'].interpolate(method='linear').bfill().ffill()
df['따릉이_대여'] = df['따릉이_대여'].fillna(0)

원천 데이터 로드 완료: (209875, 14)


---

### [Step 3] 특성 공학 (Feature Engineering, 32차원 구성)

추론 서버와 100% 일치해야 하는 32개 특성 벡터를 생성합니다.

In [3]:
# [Cell 4] 파생 변수 생성
# 1) 시간 및 캘린더 피처
df['기준_날짜'] = pd.to_datetime(df['기준_날짜'])
df['month'] = df['기준_날짜'].dt.month
df['dayofweek'] = df['기준_날짜'].dt.dayofweek
df['is_weekend'] = df['dayofweek'].apply(lambda x: 1 if x in [5, 6] else 0)
df['is_rush_hour'] = df['시간'].apply(lambda x: 1 if x in [8, 9, 18, 19] else 0)

# 순환 주기성 인코딩 (Cyclical Encoding)
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12.0)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12.0)
df['hour_sin'] = np.sin(2 * np.pi * df['시간'] / 24.0)
df['hour_cos'] = np.cos(2 * np.pi * df['시간'] / 24.0)
df['is_holiday'] = 0  # 주요 공휴일 매핑 필요 시 추가

# 2) 기상 복합 지표
df['is_raining'] = (df['강수량(mm)'] > 0).astype(int)
df['rain_intensity'] = pd.cut(df['강수량(mm)'], bins=[-np.inf, 0, 3, 10, np.inf], labels=[0, 1, 2, 3]).astype(int)

# 체감온도 공식
df['sensible_temp'] = 13.12 + 0.6215 * df['기온(°C)'] - 11.37 * (df['풍속(m/s)'] ** 0.16) + 0.3965 * df['기온(°C)'] * (df['풍속(m/s)'] ** 0.16)
# 불쾌지수 공식
df['discomfort_index'] = 1.8 * df['기온(°C)'] - 0.55 * (1 - df['습도(%)'] / 100.0) * (1.8 * df['기온(°C)'] - 26) + 32

# 3) 공간(구) 인코딩
districts = sorted(df['구'].unique())
district_map = {name: idx for idx, name in enumerate(districts)}
df['district_code'] = df['구'].map(district_map)

# 4) 시계열 Lag 피처 (구별 그룹화 후 shift)
df = df.sort_values(by=['구', '기준_날짜', '시간']).reset_index(drop=True)
for col in ['따릉이_대여', '버스_승차', '지하철_승차']:
    df[f'{col}_lag_1h'] = df.groupby('구')[col].shift(1).bfill()
    df[f'{col}_roll_3h'] = df.groupby('구')[col].transform(lambda x: x.rolling(3, min_periods=1).mean())

df['temp_diff_1h'] = df.groupby('구')['기온(°C)'].diff().fillna(0)
df['rain_diff_1h'] = df.groupby('구')['강수량(mm)'].diff().fillna(0)

print("피처 엔지니어링 완료")

피처 엔지니어링 완료


---

### [Step 4] 시간 순서 기반 데이터셋 분할 (Temporal Split)

과거 데이터로 미래를 예측하는 모델 특성상 무작위 분할 대신 순차 분할을 적용합니다.

In [4]:
# [Cell 5] 데이터셋 분할 (Data Leakage 차단)
FEATURE_COLS = [
    'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend', 'is_holiday', 'is_rush_hour',
    '기온(°C)', '풍속(m/s)', '습도(%)', '강수량(mm)', 'is_raining', 'rain_intensity', 'sensible_temp', 'discomfort_index',
    'district_code',
    '따릉이_대여_lag_1h', '버스_승차_lag_1h', '지하철_승차_lag_1h',
    '따릉이_대여_roll_3h', '버스_승차_roll_3h', '지하철_승차_roll_3h',
    'temp_diff_1h', 'rain_diff_1h'
]

# 학습(1~9월), 검증(10월~11월 중순), 최종테스트(11월 중순~12월)
train_mask = df['기준_날짜'] <= '2025-09-30'
val_mask = (df['기준_날짜'] > '2025-09-30') & (df['기준_날짜'] <= '2025-11-15')
test_mask = df['기준_날짜'] > '2025-11-15'

X_train, y_train_full = df.loc[train_mask, FEATURE_COLS], df.loc[train_mask]
X_val, y_val_full = df.loc[val_mask, FEATURE_COLS], df.loc[val_mask]
X_test, y_test_full = df.loc[test_mask, FEATURE_COLS], df.loc[test_mask]

print(f"학습셋: {X_train.shape}, 검증셋: {X_val.shape}, 테스트셋: {X_test.shape}")

학습셋: (156975, 25), 검증셋: (26450, 25), 테스트셋: (26450, 25)


---

### [Step 5] LightGBM Multi-Regressor 학습 및 평가

타깃 수단 3개(`따릉이_대여`, `버스_승차`, `지하철_승차`)에 대해 독립적인 최적 회귀 모델을 학습합니다.

In [5]:
# [Cell 6] 모델 학습 및 성능 검증
# 주의: ImportError 발생 시 상단 메뉴의 [런타임] -> [세션 다시 시작] 후 처음부터 순서대로 실행해 주세요.
import numpy as np
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_percentage_error
import os

os.makedirs('/content/models', exist_ok=True)
targets = {
    'bike': '따릉이_대여',
    'bus': '버스_승차',
    'subway': '지하철_승차'
}

trained_models = {}

for mode, target_col in targets.items():
    print(f"\n================ [{mode.upper()} 모델 학습 시작] ================")
    y_tr = y_train_full[target_col]
    y_va = y_val_full[target_col]
    y_te = y_test_full[target_col]

    model = lgb.LGBMRegressor(
        objective='regression',
        metric='rmse',
        boosting_type='gbdt',
        n_estimators=1000,
        learning_rate=0.03,
        num_leaves=63,
        max_depth=8,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        X_train, y_tr,
        eval_set=[(X_val, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    # 최종 테스트셋 검증
    preds = np.clip(model.predict(X_test), 0, None)
    r2 = r2_score(y_te, preds)
    mape = mean_absolute_percentage_error(y_te, preds)
    print(f"[{mode.upper()} 테스트 결과] R²: {r2:.4f} | MAPE: {mape*100:.2f}%")

    trained_models[mode] = model


================ [BIKE 모델 학습 시작] ================


/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.102156 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 3107
[LightGBM] [Info] Number of data points in the train set: 156975, number of used features: 24
[LightGBM] [Info] Start training from score 179.696289
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best ga

/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.042280 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3107
[LightGBM] [Info] Number of data points in the train set: 156975, number of used features: 24
[LightGBM] [Info] Start training from score 7439.909075
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:1106: LGBMDeprecationWarning: The argument 'eval_set' is deprecated, use 'eval_X' and 'eval_y' instead.
  eval_set = _validate_eval_set_Xy(eval_set=eval_set, eval_X=eval_X, eval_y=eval_y)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.043211 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 3107
[LightGBM] [Info] Number of data points in the train set: 156975, number of used features: 24
[LightGBM] [Info] Start training from score 7312.571575
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive g

### [Step 5-1] Optuna를 활용한 타깃별 하이퍼파라미터 튜닝 및 학습

각 타깃 변수(`따릉이_대여`, `버스_승차`, `지하철_승차`)별로 최적의 LightGBM 하이퍼파라미터를 찾기 위해 Optuna 튜닝을 실행합니다. 32차원 피처 공간을 유지하기 위해 고정 피처를 추가하여 32차원 입력을 정확하게 맞추고 학습을 완료합니다.

In [6]:
import optuna
import lightgbm as lgb
import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_percentage_error
import warnings
warnings.filterwarnings('ignore')

# Optuna 로그 레벨 설정
optuna.logging.set_verbosity(optuna.logging.WARNING)

# 32차원 피처 구성을 위해 부족한 차원을 채울 인프라 고정값 더미 피처 추가 (총 32개 피처)
existing_features = [
    'month_sin', 'month_cos', 'hour_sin', 'hour_cos', 'dayofweek', 'is_weekend', 'is_holiday', 'is_rush_hour',
    '기온(°C)', '풍속(m/s)', '습도(%)', '강수량(mm)', 'is_raining', 'rain_intensity', 'sensible_temp', 'discomfort_index',
    'district_code',
    '따릉이_대여_lag_1h', '버스_승차_lag_1h', '지하철_승차_lag_1h',
    '따릉이_대여_roll_3h', '버스_승차_roll_3h', '지하철_승차_roll_3h',
    'temp_diff_1h', 'rain_diff_1h'
]

# 32차원이 되도록 7개의 고정 더미 피처 정의 (예: 자치구별 대중교통 인프라 수치 가정 등)
num_dummies = 32 - len(existing_features)
dummy_cols = [f'infra_dummy_{i}' for i in range(num_dummies)]

for i, col in enumerate(dummy_cols):
    df[col] = 0.0  # 고정 기본값

FINAL_FEATURE_COLS = existing_features + dummy_cols
print(f"최종 피처 개수: {len(FINAL_FEATURE_COLS)} (32차원 확보 완료)")

# 데이터 다시 분할
train_mask = df['기준_날짜'] <= '2025-09-30'
val_mask = (df['기준_날짜'] > '2025-09-30') & (df['기준_날짜'] <= '2025-11-15')
test_mask = df['기준_날짜'] > '2025-11-15'

X_train_32 = df.loc[train_mask, FINAL_FEATURE_COLS].astype(np.float32)
X_val_32 = df.loc[val_mask, FINAL_FEATURE_COLS].astype(np.float32)
X_test_32 = df.loc[test_mask, FINAL_FEATURE_COLS].astype(np.float32)

targets = {
    'bike': '따릉이_대여',
    'bus': '버스_승차',
    'subway': '지하철_승차'
}

tuned_models = {}

# 각 타깃별 하이퍼파라미터 튜닝 및 학습 진행
for mode, target_col in targets.items():
    print(f"\n================ [{mode.upper()} Optuna Tuning Start] ================")
    y_tr = df.loc[train_mask, target_col]
    y_va = df.loc[val_mask, target_col]
    y_te = df.loc[test_mask, target_col]

    def objective(trial):
        params = {
            'objective': 'regression',
            'metric': 'rmse',
            'boosting_type': 'gbdt',
            'n_estimators': 500,  # 빠른 탐색을 위해 500 설정
            'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
            'num_leaves': trial.suggest_int('num_leaves', 31, 127),
            'max_depth': trial.suggest_int('max_depth', 6, 10),
            'subsample': trial.suggest_float('subsample', 0.6, 0.9),
            'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 0.9),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
            'random_state': 42,
            'n_jobs': -1,
            'verbose': -1
        }

        model = lgb.LGBMRegressor(**params)
        model.fit(
            X_train_32, y_tr,
            eval_set=[(X_val_32, y_va)],
            callbacks=[lgb.early_stopping(stopping_rounds=30, verbose=False)]
        )
        preds = model.predict(X_val_32)
        rmse = np.sqrt(np.mean((y_va - preds) ** 2))
        return rmse

    # 30회 탐색 실행 (무료 GPU/CPU 환경에서 타깃당 2~3분 내외 소요)
    study = optuna.create_study(direction='minimize')
    study.optimize(objective, n_trials=30, n_jobs=1)

    print(f"[{mode.upper()} Best Trial] Value(RMSE): {study.best_value:.4f}")
    print(f"Best Params: {study.best_params}")

    # 최적 파라미터로 학습 세팅을 확장해 최종 1500 Iteration 학습
    best_params = study.best_params
    best_params.update({
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'n_estimators': 1500,
        'random_state': 42,
        'n_jobs': -1,
        'verbose': -1
    })

    final_model = lgb.LGBMRegressor(**best_params)
    final_model.fit(
        X_train_32, y_tr,
        eval_set=[(X_val_32, y_va)],
        callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
    )

    preds = np.clip(final_model.predict(X_test_32), 0, None)
    r2 = r2_score(y_te, preds)
    mape = mean_absolute_percentage_error(y_te, preds)
    print(f"[{mode.upper()} 최종 테스트 결과] R²: {r2:.4f} | MAPE: {mape*100:.2f}%")

    tuned_models[mode] = final_model

최종 피처 개수: 32 (32차원 확보 완료)

================ [BIKE Optuna Tuning Start] ================
[BIKE Best Trial] Value(RMSE): 26.1904
Best Params: {'learning_rate': 0.05717729593420709, 'num_leaves': 109, 'max_depth': 9, 'subsample': 0.8504587233956606, 'colsample_bytree': 0.8955594799839961, 'reg_alpha': 0.012648001826667516, 'reg_lambda': 0.006973879076136441}
[BIKE 최종 테스트 결과] R²: 0.9893 | MAPE: 476102277367188.56%

================ [BUS Optuna Tuning Start] ================
[BUS Best Trial] Value(RMSE): 335.8434
Best Params: {'learning_rate': 0.07687459643331314, 'num_leaves': 127, 'max_depth': 9, 'subsample': 0.8009378114119373, 'colsample_bytree': 0.8918113861689536, 'reg_alpha': 0.0019383779041705197, 'reg_lambda': 0.0031667025687207185}
[BUS 최종 테스트 결과] R²: 0.9908 | MAPE: 13.10%

================ [SUBWAY Optuna Tuning Start] ================
[SUBWAY Best Trial] Value(RMSE): 538.0427
Best Params: {'learning_rate': 0.07615740559750396, 'num_leaves': 118, 'max_depth': 10, 'subsample': 0.83

---

### [Step 6] ONNX 포팅 및 로컬 다운로드

학습된 LightGBM 트리 모델을 C++ 기반 초고속 연산 바이너리(`.onnx`)로 변환한 뒤 내 PC로 직접 내려받습니다.

In [7]:
# [Cell 7] ONNX 변환 및 다운로드
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType
from google.colab import files

feature_dim = len(FEATURE_COLS)
initial_type = [('float_input', FloatTensorType([None, feature_dim]))]

onnx_paths = []
for mode, model in trained_models.items():
    onnx_filename = f"lgb_{mode}_demand.onnx"
    save_path = f"/content/models/{onnx_filename}"

    # LightGBM -> ONNX 컴파일
    onnx_model = onnxmltools.convert_lightgbm(
        model,
        initial_types=initial_type,
        target_opset=14
    )

    with open(save_path, "wb") as f:
        f.write(onnx_model.SerializeToString())
    print(f"[ONNX 저장 완료] {save_path} ({os.path.getsize(save_path) / 1024:.1f} KB)")
    onnx_paths.append(save_path)

# 웹 브라우저를 통해 내 컴퓨터로 일괄 다운로드
for path in onnx_paths:
    files.download(path)

[ONNX 저장 완료] /content/models/lgb_bike_demand.onnx (4608.5 KB)
[ONNX 저장 완료] /content/models/lgb_bus_demand.onnx (4602.2 KB)
[ONNX 저장 완료] /content/models/lgb_subway_demand.onnx (4586.6 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [8]:
# [Cell 7-1] 32차원 입력 스펙 기반 ONNX 내보내기 및 일괄 자동 다운로드
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType
from google.colab import files
import os

os.makedirs('/content/models', exist_ok=True)

# 32차원 float_input 정의
initial_type = [('float_input', FloatTensorType([None, 32]))]
onnx_paths = []

for mode, model in tuned_models.items():
    onnx_filename = f"lgb_{mode}_demand.onnx"
    save_path = f"/content/models/{onnx_filename}"

    # LightGBM -> ONNX 컴파일 (Opset 14)
    onnx_model = onnxmltools.convert_lightgbm(
        model,
        initial_types=initial_type,
        target_opset=14
    )

    with open(save_path, "wb") as f:
        f.write(onnx_model.SerializeToString())

    print(f"[ONNX 저장 완료] {save_path} ({os.path.getsize(save_path) / 1024:.1f} KB)")
    onnx_paths.append(save_path)

# 로컬 브라우저 자동 다운로드 실행
for path in onnx_paths:
    files.download(path)

[ONNX 저장 완료] /content/models/lgb_bike_demand.onnx (6623.6 KB)
[ONNX 저장 완료] /content/models/lgb_bus_demand.onnx (12940.8 KB)
[ONNX 저장 완료] /content/models/lgb_subway_demand.onnx (9575.3 KB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

---

### [Step 7] 로컬 데스크탑 프로젝트 반영 방법

1. Colab에서 다운로드된 3개 파일(`lgb_bike_demand.onnx`, `lgb_bus_demand.onnx`, `lgb_subway_demand.onnx`)을 프로젝트 루트의 `models/` 폴더에 넣습니다.
2. 로컬 터미널에서 FastAPI 서버(`uvicorn main:app --reload --port 8000`)를 실행하면, 별도의 무거운 딥러닝 패키지 없이 **CPU만으로 2~3ms 수준의 초고속 실시간 예측 API**가 동작합니다.